In [6]:
import sagemaker
import boto3

try:
    from sagemaker.sklearn.estimator import SKLearn
    from sagemaker.estimator import Estimator
except Exception:
    SKLearn = None
    Estimator = None

import pandas as pd
import numpy as np
import math
import json
import os
import pprint
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

mpl.rcParams['figure.figsize'] = (15.0, 6.0)
pd.set_option('display.max_columns', 30)

In [7]:
# Create execution context (SageMaker if available, otherwise local mode)
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
LOCAL_MODE = False

try:
    sagemaker_session = sagemaker.Session()
    role = sagemaker.get_execution_role()
    print('Running in SageMaker mode')
except Exception as err:
    LOCAL_MODE = True
    sagemaker_session = None
    role = None
    print(f'Running in local mode: {err}')

Running in local mode: module 'sagemaker' has no attribute 'Session'


## Upload Trainings Data To S3

In [8]:
# Read preprocessed data
data = (
    pd.read_csv(DATA_DIR / 'processed' / 'preprocessed_data.csv')
    .rename(columns={'offer_completed': 'label'})
    .set_index(['person', 'offer'])
)
data.head()

label  \
person                           offer                                     
0009655768c64bdeb2e877511632db8f f19421c1d4aa40978ebb69ca19b0e20d   True   
                                 fafdcd668e3743c1bb461111dcafc2a4   True   
00116118485d4dfda04fdbaba9a87b5c f19421c1d4aa40978ebb69ca19b0e20d  False   
0011e0d4e6b944f998e987f904e8c1e5 0b1e1539f2cc45b7b9fa7c272da2e1d7   True   
                                 2298d6c36e964ae4a3e7e9706d1fb8c2   True   

                                                                    age  \
person                           offer                                    
0009655768c64bdeb2e877511632db8f f19421c1d4aa40978ebb69ca19b0e20d  33.0   
                                 fafdcd668e3743c1bb461111dcafc2a4  33.0   
00116118485d4dfda04fdbaba9a87b5c f19421c1d4aa40978ebb69ca19b0e20d   NaN   
0011e0d4e6b944f998e987f904e8c1e5 0b1e1539f2cc45b7b9fa7c272da2e1d7  40.0   
                                 2298d6c36e964ae4a3e7e9706d1fb8c2  40.0   

                                                                    income  \
person                           offer                                       
0009655768c64bdeb2e877511632db8f f19421c1d4aa40978ebb69ca19b0e20d  72000.0   
                                 fafdcd668e3743c1bb461111dcafc2a4  72000.0   
00116118485d4dfda04fdbaba9a87b5c f19421c1d4aa40978ebb69ca19b0e20d      NaN   
0011e0d4e6b944f998e987f904e8c1e5 0b1e1539f2cc45b7b9fa7c272da2e1d7  57000.0   
                                 2298d6c36e964ae4a3e7e9706d1fb8c2  57000.0   

                                                                   reward  \
person                           offer                                      
0009655768c64bdeb2e877511632db8f f19421c1d4aa40978ebb69ca19b0e20d       5   
                                 fafdcd668e3743c1bb461111dcafc2a4       2   
00116118485d4dfda04fdbaba9a87b5c f19421c1d4aa40978ebb69ca19b0e20d       5   
0011e0d4e6b944f998e987f904e8c1e5 0b1e1539f2cc45b7b9fa7c272da2e1d7       5   
                                 2298d6c36e964ae4a3e7e9706d1fb8c2       3   

                                                                   difficulty  \
person                           offer                                          
0009655768c64bdeb2e877511632db8f f19421c1d4aa40978ebb69ca19b0e20d           5   
                                 fafdcd668e3743c1bb461111dcafc2a4          10   
00116118485d4dfda04fdbaba9a87b5c f19421c1d4aa40978ebb69ca19b0e20d           5   
0011e0d4e6b944f998e987f904e8c1e5 0b1e1539f2cc45b7b9fa7c272da2e1d7          20   
                                 2298d6c36e964ae4a3e7e9706d1fb8c2           7   

                                                                   duration  \
person                           offer                                        
0009655768c64bdeb2e877511632db8f f19421c1d4aa40978ebb69ca19b0e20d         5   
                                 fafdcd668e3743c1bb461111dcafc2a4        10   
00116118485d4dfda04fdbaba9a87b5c f19421c1d4aa40978ebb69ca19b0e20d         5   
0011e0d4e6b944f998e987f904e8c1e5 0b1e1539f2cc45b7b9fa7c272da2e1d7        10   
                                 2298d6c36e964ae4a3e7e9706d1fb8c2         7   

                                                                   days_since_registration  \
person                           offer                                                       
0009655768c64bdeb2e877511632db8f f19421c1d4aa40978ebb69ca19b0e20d                      461   
                                 fafdcd668e3743c1bb461111dcafc2a4                      461   
00116118485d4dfda04fdbaba9a87b5c f19421c1d4aa40978ebb69ca19b0e20d                       92   
0011e0d4e6b944f998e987f904e8c1e5 0b1e1539f2cc45b7b9fa7c272da2e1d7                      198   
                                 2298d6c36e964ae4a3e7e9706d1fb8c2                      198   

                                                                   web  email  \
person                           offer                     

In [9]:
# Make train test split
train, test = train_test_split(data, test_size=0.3, random_state=0)
print("train shape: ", train.shape)
print("test shape: ", test.shape)

train shape:  (27878, 14)
test shape:  (11948, 14)


In [10]:
# Prepare training artifacts and upload to S3 only when SageMaker is available
data_dir = DATA_DIR
data_dir.mkdir(parents=True, exist_ok=True)
bucket = sagemaker_session.default_bucket() if not LOCAL_MODE else 'local-bucket'
prefix = 'sagemaker/starbucks_rewards'

train_path = data_dir / 'train.csv'
test_path = data_dir / 'test.csv'
train.to_csv(train_path, header=False, index=False)
test.to_csv(test_path, header=False, index=False)

if LOCAL_MODE:
    train_location = str(train_path)
    test_location = str(test_path)
else:
    train_location = sagemaker_session.upload_data(str(train_path), key_prefix=prefix)
    test_location = sagemaker_session.upload_data(str(test_path), key_prefix=prefix)

In [11]:
# Check upload status
if LOCAL_MODE:
    print(f'Local train artifact: {train_location}')
    print(f'Local test artifact: {test_location}')
else:
    s3_client = boto3.client('s3')
    for obj in s3_client.list_objects(Bucket=bucket)['Contents']:
        print(obj['Key'])

Local train artifact: d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_data_driven_marketing\starbucks_reward_data_marketing\data\train.csv
Local test artifact: d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_data_driven_marketing\starbucks_reward_data_marketing\data\test.csv


## Train Estimator

In [12]:
train_script_path = PROJECT_ROOT / 'src' / 'train.py'
print(train_script_path)
print(train_script_path.read_text(encoding='utf-8')[:2000])

d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_data_driven_marketing\starbucks_reward_data_marketing\src\train.py
import argparse
import os
import pandas as pd
import numpy as np

import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV

# Define hyperparameter space
HYPERPARAMETER_GRID = {
    'imputer__strategy': ['mean', 'median'],
    'rf__bootstrap': [True, False],
    'rf__max_depth': [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, None],
    'rf__max_features': ['auto', 'sqrt'],
    'rf__min_samples_leaf': [1, 2, 4],
    'rf__min_samples_split': [2, 5, 10],
    'rf__n_estimators': [200, 400, 600, 800, 1000, 1200, 1400, 1600, 1800, 2000]    
}


def model_fn(model_dir):
    """Load model from the model_dir. This is the same model that is saved
    in the main if s

In [13]:
# Set directory to save model artifacts
s3_output_path = f"s3://{bucket}/{prefix}/output"

# Instantiate the sklearn estimator (SageMaker mode only)
if LOCAL_MODE:
    estimator = None
else:
    estimator = SKLearn(
        sagemaker_session=sagemaker_session,
        role=role,
        entry_point='train.py',
        source_dir='src',
        py_version='py3',
        framework_version='0.23-1',
        instance_count=1,
        instance_type='ml.c4.xlarge',
        output_path=s3_output_path
    )

In [14]:
%%time

# Train model in SageMaker or local fallback
if LOCAL_MODE:
    X_train_local = train.iloc[:, 1:].copy()
    y_train_local = train.iloc[:, 0].copy()
    local_model = RandomForestClassifier(n_estimators=400, random_state=42)
    local_model.fit(X_train_local, y_train_local)
else:
    estimator.fit({'train': train_location})

CPU times: total: 7.12 s
Wall time: 7.19 s


## Deploy Predictive Model

In [15]:
%%time

# Deploy model and assign predictor
if LOCAL_MODE:
    class LocalPredictor:
        def __init__(self, model):
            self.model = model

        def predict(self, X):
            X_df = pd.DataFrame(X) if not isinstance(X, (pd.DataFrame, np.ndarray)) else X
            return self.model.predict_proba(X_df)[:, 1]

        def delete_endpoint(self):
            print('Local mode: no endpoint to delete.')

    predictor = LocalPredictor(local_model)
else:
    predictor = estimator.deploy(
        initial_instance_count=1,
        instance_type='ml.t2.medium'
    )

CPU times: total: 0 ns
Wall time: 0 ns


## Evaluate Predictive Model

In [16]:
# Split test data into feature matrix and target vector
X_test = test.iloc[:,1:].copy()
y_test = test.iloc[:,0].copy()

In [17]:
pred_proba = predictor.predict(X_test)
pred_proba

array([0.36  , 0.6675, 0.    , ..., 0.    , 0.38  , 0.9475],
      shape=(11948,))

In [18]:
roc_auc_score(y_test, pred_proba)

0.8073138623987786

## Recommendation System

### Illustrate API Usage

In [19]:
customers = [
    {
        'id': '0009655768c64bdeb2e877511632db8f',
        'age': 33,
        'gender': 'M',
        'income': 72000,
        'days_since_registration': 461}, 
    {
        'id': '00116118485d4dfda04fdbaba9a87b5c',
        'age': np.nan,
        'gender': np.nan,
        'income': np.nan,
        'days_since_registration': 92}, 
    {
        'id': '0011e0d4e6b944f998e987f904e8c1e5',
        'age': 40,
        'gender': 'O',
        'income': 57000,
        'days_since_registration': 198}
]

In [20]:
offers = [
    {
        'id': 'ae264e3637204a6fb9bb56bc8210ddfd',
        'type': 'bogo',
        'web': 1,
        'email': 1,
        'social': 1,
        'mobile': 0,
        'reward': 10,
        'difficulty': 10,
        'duration': 7}, 
    {
        'id': 'f19421c1d4aa40978ebb69ca19b0e20d',
        'type': 'bogo',
        'web': 1,
        'email': 1,
        'social': 1,
        'mobile': 1,
        'reward': 5,
        'difficulty': 5,
        'duration': 5}, 
    {
        'id': '0b1e1539f2cc45b7b9fa7c272da2e1d7',
        'type': 'discount',
        'web': 1,
        'email': 1,
        'social': 0,
        'mobile': 0,
        'reward': 5,
        'difficulty': 20,
        'duration': 10}  
]

In [21]:
def get_success_probabilities(customers, offers):
    """Calculates for each customer the success probabilities of various offers"""
    
    probas = {}
    for customer in customers:
        probas[customer['id']] = {}

        for offer in offers:
            pred = predictor.predict([[
                customer['age'], customer['income'], offer['reward'], offer['difficulty'], offer['duration'], 
                customer['days_since_registration'], offer['web'], offer['email'], offer['mobile'], offer['social'], 
                (1 if customer['gender']=='M' else 0), (1 if customer['gender']=='O' else 0), (1 if offer['type']=='discount' else 0)
            ]])
            probas[customer['id']][offer['id']] = float(pred.squeeze())
            
    return probas

def get_best_offer_for_customer(probas):
    """Infers best offer for customer from success probabilities"""
    
    choices = {}
    for customer in probas.keys():
        
        best_offer = None
        best_proba = 0
        for offer_id, proba in probas[customer].items():
            if proba > best_proba:
                best_offer = offer_id
                best_proba = proba
            
        choices[customer] = best_offer
        
    return choices

In [22]:
probas = get_success_probabilities(customers, offers)
pprint.pprint(probas)

d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_data_driven_marketing\starbucks_reward_data_marketing\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_data_driven_marketing\starbucks_reward_data_marketing\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_data_driven_marketing\starbucks_reward_data_marketing\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\Pow

{'0009655768c64bdeb2e877511632db8f': {'0b1e1539f2cc45b7b9fa7c272da2e1d7': 0.965,
                                      'ae264e3637204a6fb9bb56bc8210ddfd': 0.5025,
                                      'f19421c1d4aa40978ebb69ca19b0e20d': 0.98},
 '00116118485d4dfda04fdbaba9a87b5c': {'0b1e1539f2cc45b7b9fa7c272da2e1d7': 0.0,
                                      'ae264e3637204a6fb9bb56bc8210ddfd': 0.0,
                                      'f19421c1d4aa40978ebb69ca19b0e20d': 0.0},
 '0011e0d4e6b944f998e987f904e8c1e5': {'0b1e1539f2cc45b7b9fa7c272da2e1d7': 0.9,
                                      'ae264e3637204a6fb9bb56bc8210ddfd': 0.5325,
                                      'f19421c1d4aa40978ebb69ca19b0e20d': 0.6675}}


In [23]:
choices = get_best_offer_for_customer(probas)
pprint.pprint(choices)

{'0009655768c64bdeb2e877511632db8f': 'f19421c1d4aa40978ebb69ca19b0e20d',
 '00116118485d4dfda04fdbaba9a87b5c': None,
 '0011e0d4e6b944f998e987f904e8c1e5': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}


### Measure Completion Rate Uplift

In [24]:
# Append prediction probabilities to test set
test['prediction'] = pred_proba

In [25]:
# How many users have more than one offer?
offers_per_customer = test.reset_index().person.value_counts()
np.sum(offers_per_customer > 1)

np.int64(2489)

In [26]:
# Exclude users with only one offer from test set
test.reset_index(inplace=True)
test_more_than_1_offer = test[test.person.isin(offers_per_customer[offers_per_customer>1].index)].copy()
test_more_than_1_offer.shape

(5355, 17)

In [27]:
# Completion rate of recommendation system
(
    test_more_than_1_offer.reset_index()
    .groupby('person')
    .apply(lambda df: df.sort_values('prediction', ascending=False).head(1))
    .label.mean()
)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6336\2748824048.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: df.sort_values('prediction', ascending=False).head(1))


np.float64(0.657292085174769)

In [28]:
# Completion rate of random model
(
    test_more_than_1_offer.reset_index()
    .groupby('person')
    .apply(lambda df: df.sample(1))
    .label.mean()
)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6336\549143351.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: df.sample(1))


np.float64(0.6054640417838489)

## Delete Endpoint

In [29]:
predictor.delete_endpoint()

Local mode: no endpoint to delete.
